1. Setup

In [ ]:
import sys
from pathlib import Path

sys.path.insert(
    0,
    str(Path.cwd().parent)
)

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold import TSNE

import time

from langchain_openai import OpenAIEmbeddings

from utils.config import config

plt.style.use(
    'seaborn-v0_8-whitegrid'
)

sns.set_palette('Set2')

%matplotlib inline

# Requires OpenAI API Key
print(
    f"Using OpenAI API Key: "
    f"{config.openai_api_key[:20]}..."
)

2. Sample Texts

In [ ]:
sample_texts = [

    # AI / ML
    "Machine learning is a subset of artificial intelligence that focuses on learning from data.",

    "Artificial intelligence includes machine learning, which learns patterns from datasets.",

    "Deep learning is a type of machine learning that uses neural networks with multiple layers.",

    # Programming

    "Python is a popular programming language used for web development and data science.",

    "JavaScript is widely used for web development and creating interactive websites.",

    "Java is an object-oriented programming language used for enterprise applications.",

    # Other topics

    "Climate change is causing rising temperatures and extreme weather patterns globally.",

    "Renewable energy sources like solar and wind power are becoming more affordable.",

    "Quantum computing uses quantum mechanics to process information in new ways.",

    "Blockchain technology enables secure, decentralized record-keeping systems."
]

text_labels = [
    "AI/ML-1",
    "AI/ML-2",
    "AI/ML-3",
    "Programming-1",
    "Programming-2",
    "Programming-3",
    "Climate",
    "Energy",
    "Quantum",
    "Blockchain"
]

print("Sample texts loaded:\n")

for i, (label, text) in enumerate(
    zip(text_labels, sample_texts)
):
    print(
        f"{i+1}. [{label}] "
        f"{text[:60]}..."
    )

3. Embedding Models Comparison

In [ ]:
embedding_models = {

    "text-embedding-3-small":
    OpenAIEmbeddings(
        model="text-embedding-3-small"
    ),

    "text-embedding-3-large":
    OpenAIEmbeddings(
        model="text-embedding-3-large"
    ),

    "text-embedding-ada-002":
    OpenAIEmbeddings(
        model="text-embedding-ada-002"
    )
}

print(
    "Embedding models initialized:"
)

for model_name in embedding_models:
    print(f"  • {model_name}")

4. Generate Embeddings

In [ ]:
embeddings_dict = {}

timing_results = []

for model_name, embedder in embedding_models.items():

    print(
        f"\nGenerating embeddings with "
        f"{model_name}..."
    )

    start_time = time.time()

    embeddings = embedder.embed_documents(
        sample_texts
    )

    elapsed = (
        time.time() - start_time
    )

    embeddings_dict[
        model_name
    ] = np.array(embeddings)

    timing_results.append({
        "model": model_name,
        "time": elapsed,
        "dimension":
            len(embeddings[0]),
        "texts":
            len(sample_texts)
    })

    print(
        f"✓ Dimension: "
        f"{len(embeddings[0])}"
    )

    print(
        f"✓ Time: "
        f"{elapsed:.3f}s"
    )

df_timing = pd.DataFrame(
    timing_results
)

display(df_timing)

5. Similarity Analysis

Generate Similarity Matrices

In [ ]:
similarity_matrices = {}

for model_name, embeddings in embeddings_dict.items():

    similarity_matrices[
        model_name
    ] = cosine_similarity(
        embeddings
    )

    df_sim = pd.DataFrame(
        similarity_matrices[
            model_name
        ],
        index=text_labels,
        columns=text_labels
    )

    print(
        f"\n{model_name} "
        f"Similarity Matrix"
    )

    display(
        df_sim.round(3)
    )

Heatmap Visualization

In [ ]:
fig, axes = plt.subplots(
    1,
    3,
    figsize=(18, 5)
)

for idx, (
    model_name,
    sim_matrix
) in enumerate(
    similarity_matrices.items()
):

    sns.heatmap(
        sim_matrix,
        xticklabels=text_labels,
        yticklabels=text_labels,
        annot=True,
        fmt=".2f",
        cmap="RdYlGn",
        vmin=0,
        vmax=1,
        square=True,
        ax=axes[idx]
    )

    axes[idx].set_title(
        f"{model_name}\n"
        f"Similarity Matrix"
    )

plt.tight_layout()
plt.show()

6. Clustering Analysis (t-SNE)

In [ ]:
fig, axes = plt.subplots(
    1,
    3,
    figsize=(18, 5)
)

category_colors = {
    "AI/ML": "red",
    "Programming": "blue",
    "Other": "green"
}

def get_category(label):

    if "AI/ML" in label:
        return "AI/ML"

    elif "Programming" in label:
        return "Programming"

    return "Other"

colors = [
    category_colors[
        get_category(label)
    ]
    for label in text_labels
]

for idx, (
    model_name,
    embeddings
) in enumerate(
    embeddings_dict.items()
):

    tsne = TSNE(
        n_components=2,
        random_state=42,
        perplexity=3
    )

    embeddings_2d = (
        tsne.fit_transform(
            embeddings
        )
    )

    for i, (x, y) in enumerate(
        embeddings_2d
    ):

        axes[idx].scatter(
            x,
            y,
            c=colors[i],
            s=200,
            alpha=0.6
        )

        axes[idx].annotate(
            text_labels[i],
            (x, y),
            fontsize=8
        )

    axes[idx].set_title(
        f"{model_name}\n"
        f"t-SNE Visualization"
    )

plt.tight_layout()
plt.show()

7. Semantic Search Quality Test

In [ ]:
test_queries = [
    "What is artificial intelligence?",
    "Tell me about programming languages",
    "Explain renewable energy"
]

expected_matches = [
    [0, 1, 2],
    [3, 4, 5],
    [7]
]

search_results = []

for model_name, embedder in embedding_models.items():

    print("\n" + "=" * 80)
    print(model_name)
    print("=" * 80)

    for query_idx, query in enumerate(
        test_queries
    ):

        query_embedding = (
            embedder.embed_query(query)
        )

        similarities = cosine_similarity(
            [query_embedding],
            embeddings_dict[
                model_name
            ]
        )[0]

        top_indices = np.argsort(
            similarities
        )[-3:][::-1]

        print(f"\nQuery: {query}")

        for rank, idx in enumerate(
            top_indices,
            start=1
        ):

            match = (
                "✓"
                if idx in expected_matches[
                    query_idx
                ]
                else "✗"
            )

            print(
                f"{rank}. {match} "
                f"{text_labels[idx]}"
            )

        hits = sum(
            1
            for idx in top_indices
            if idx in expected_matches[
                query_idx
            ]
        )

        precision = (
            hits /
            len(top_indices)
        )

        search_results.append({
            "model": model_name,
            "query": query,
            "precision": precision
        })

8. Search Quality Visualization

In [ ]:
df_search = pd.DataFrame(
    search_results
)

df_search_pivot = (
    df_search.pivot(
        index="query",
        columns="model",
        values="precision"
    )
)

display(df_search_pivot)

df_search_pivot.plot(
    kind="bar",
    figsize=(12, 6),
    rot=0
)

plt.title(
    "Search Quality Comparison (Precision@3)"
)

plt.xlabel("Query")
plt.ylabel("Precision")

plt.ylim(0, 1.1)

plt.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()
plt.show()

9. Model Comparison Summary

In [ ]:
model_summary = []

for model_name in embedding_models:

    timing = df_timing[
        df_timing["model"]
        == model_name
    ].iloc[0]

    avg_precision = (
        df_search[
            df_search["model"]
            == model_name
        ]["precision"]
        .mean()
    )

    sim_matrix = (
        similarity_matrices[
            model_name
        ]
    )

    aiml_sim = np.mean([
        sim_matrix[0, 1],
        sim_matrix[0, 2],
        sim_matrix[1, 2]
    ])

    prog_sim = np.mean([
        sim_matrix[3, 4],
        sim_matrix[3, 5],
        sim_matrix[4, 5]
    ])

    avg_ingroup_sim = (
        aiml_sim +
        prog_sim
    ) / 2

    model_summary.append({

        "Model":
            model_name,

        "Dimension":
            int(
                timing["dimension"]
            ),

        "Avg Time (s)":
            timing["time"] /
            timing["texts"],

        "Search Precision":
            avg_precision,

        "In-group Similarity":
            avg_ingroup_sim
    })

df_summary = pd.DataFrame(
    model_summary
)

print("=" * 80)
print(
    "MODEL COMPARISON SUMMARY"
    .center(80)
)
print("=" * 80)

display(df_summary)

10. Recommendations

In [ ]:
print("""
EMBEDDING MODEL RECOMMENDATIONS

1. text-embedding-3-small
   ✓ Recommended for most projects

   Advantages:
   • Fast
   • Low cost
   • Good retrieval quality
   • 1536 dimensions

   Best Use:
   • Production RAG systems
   • Cost-sensitive deployments
   • Large-scale indexing


2. text-embedding-3-large
   ✓ Highest accuracy

   Advantages:
   • Strong semantic understanding
   • Better retrieval precision
   • 3072 dimensions

   Limitations:
   • Higher cost
   • Slower generation

   Best Use:
   • Legal search
   • Medical applications
   • Scientific research


3. text-embedding-ada-002

   Advantages:
   • Stable
   • Proven in production

   Limitations:
   • Older model
   • Generally outperformed
     by text-embedding-3-small

   Best Use:
   • Legacy systems


FINAL RECOMMENDATION

→ Production RAG:
   text-embedding-3-small

→ Maximum Accuracy:
   text-embedding-3-large
""")

11. Save Results

In [ ]:
from datetime import datetime

export_dir = (
    Path.cwd() / "exports"
)

export_dir.mkdir(
    exist_ok=True
)

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

df_summary.to_csv(
    export_dir /
    f"embedding_comparison_{timestamp}.csv",
    index=False
)

df_search.to_csv(
    export_dir /
    f"search_quality_{timestamp}.csv",
    index=False
)

print(
    f"✓ Results exported to "
    f"{export_dir}"
)